# Lab 10: Reasoning Memory

In this lab, you will record agent reasoning traces, search for similar past tasks, and view tool usage statistics. Reasoning memory captures what the agent did and whether it worked — enabling the agent to learn from past executions by reusing successful strategies and avoiding approaches that failed.

## What you will learn

- How to record agent execution traces with `record_agent_trace()`
- How to find similar past traces with `get_similar_traces()`
- How to view tool statistics with `get_tool_stats()`

## Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
assert os.getenv("NEO4J_URI"), "NEO4J_URI not set in .env file"
print("Environment loaded successfully")

Environment loaded successfully


## Import Dependencies

In [2]:
from pydantic import SecretStr

from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.integrations.microsoft_agent import (
    Neo4jMicrosoftMemory,
    record_agent_trace,
    get_similar_traces,
)

## Configure and Connect

In [3]:
settings = MemorySettings(
    neo4j={
        "uri": os.environ["NEO4J_URI"],
        "username": os.environ["NEO4J_USERNAME"],
        "password": SecretStr(os.environ["NEO4J_PASSWORD"]),
    },
)

memory_client = MemoryClient(settings)
await memory_client.connect()
print("Connected to Neo4j")

Connected to Neo4j


## Create Memory

In [4]:
memory = Neo4jMicrosoftMemory.from_memory_client(
    memory_client=memory_client,
    session_id="reasoning-demo",
    include_short_term=True,
    include_long_term=True,
    include_reasoning=True,
)

## Record an Agent Trace

Traces let the agent learn from experience — when it encounters a similar task later, it can retrieve successful past approaches and avoid strategies that failed. Use `record_agent_trace()` to capture a completed agent execution. Provide:
- `memory` — the `Neo4jMicrosoftMemory` instance
- `messages` — conversation messages (can be empty for standalone traces)
- `task` — description of what the agent was trying to accomplish
- `tool_calls` — list of tool invocations with name, arguments, result, status, and duration_ms
- `outcome` — the final result
- `success` — whether the task completed successfully

In [5]:
trace = await record_agent_trace(
    memory=memory,
    messages=[],
    task="Find sci-fi movies about time travel",
    tool_calls=[
        {
            "name": "search_knowledge",
            "arguments": {"query": "time travel sci-fi"},
            "result": ["Interstellar", "The Matrix", "Back to the Future"],
            "status": "success",
            "duration_ms": 150,
        }
    ],
    outcome="Recommended Interstellar based on user preference for Nolan films",
    success=True,
)
print(f"Recorded trace: {trace}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_STEP` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=37, offset=36>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 36, 'line': 1, 'column': 37}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (:ReasoningTrace {id: $id})-[:HAS_STEP]->(s:ReasoningStep) RETURN count(s) AS count'


Recorded trace: id=UUID('90222cd2-9992-4ef1-aad4-a1618c193b72') created_at=datetime.datetime(2026, 3, 11, 18, 4, 15, 731609) updated_at=None embedding=None metadata={} session_id='reasoning-demo' task='Find sci-fi movies about time travel' task_embedding=[-0.04949951171875, 0.0192108154296875, -0.032989501953125, -0.025421142578125, -0.0252685546875, -0.036773681640625, -0.008453369140625, -0.00649261474609375, 0.0241241455078125, -0.00890350341796875, -0.01971435546875, -0.0330810546875, -0.020599365234375, -0.01279449462890625, -0.008392333984375, -0.016357421875, -0.04095458984375, -0.04388427734375, 0.04315185546875, 0.044219970703125, -0.034759521484375, 0.03948974609375, -0.00635528564453125, -0.0043182373046875, -0.0243072509765625, -0.01117706298828125, -0.004962921142578125, 0.03363037109375, 0.0022373199462890625, 0.051177978515625, 0.03533935546875, -0.0321044921875, 0.00574493408203125, -0.006305694580078125, -0.0147247314453125, 0.00997161865234375, 0.0245361328125, 0.0243

## Find Similar Traces

Use `get_similar_traces()` to search for reasoning traces from similar past tasks. The search uses the vector embedding of the task description to find semantically similar traces.

In [6]:
traces = await get_similar_traces(
    memory=memory,
    task="Find action movies with good ratings",
    limit=3,
)

for trace in traces:
    print(f"Task: {trace.task}")
    print(f"Outcome: {trace.outcome}")
    print(f"Success: {trace.success}")
    print(f"Steps: {len(trace.steps)}")

Task: Find sci-fi movies about time travel
Outcome: Recommended Interstellar based on user preference for Nolan films
Success: True
Steps: 0


## View Tool Statistics

Reasoning memory tracks aggregate statistics about tool usage. These help identify which tools are reliable and fast vs. which fail frequently or are slow, informing decisions about tool selection and optimization.

In [7]:
stats = await memory_client.reasoning.get_tool_stats()

for tool in stats:
    print(f"{tool.name}: {tool.success_rate:.0%} success, "
          f"avg {tool.avg_duration_ms}ms")

search_knowledge: 100% success, avg 0.0ms


## Close the Connection

In [8]:
await memory_client.close()
print("Connection closed")

Connection closed


## Summary

Reasoning memory records agent execution traces including tasks, steps, tool calls, and outcomes. These traces are stored with vector embeddings so the agent can find similar past tasks using semantic search. The agent can learn from past successes and failures, either through automatic context injection or explicit tool calls.